## Sentence-BERT + Logistic Regression Accuracy

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load CLINC150 dataset
dataset = load_dataset("clinc_oos", "plus")
train_data = dataset["train"]
test_data = dataset["test"]

# Filter in-scope queries
train_in_scope = [ex for ex in train_data if ex["intent"] != 42]
test_in_scope = [ex for ex in test_data if ex["intent"] != 42]

# Load Sentence-BERT
model = SentenceTransformer('intfloat/e5-base')

# Generate embeddings
train_texts = [ex["text"] for ex in train_in_scope]
test_texts = [ex["text"] for ex in test_in_scope]
train_embeddings = model.encode(train_texts, batch_size=64, show_progress_bar=True)
test_embeddings = model.encode(test_texts, batch_size=64, show_progress_bar=True)
train_labels = [ex["intent"] for ex in train_in_scope]
test_labels = [ex["intent"] for ex in test_in_scope]
# Train logistic regression
clf = LogisticRegression(multi_class='multinomial', max_iter=1000)
clf.fit(train_embeddings, train_labels)

# Predict and evaluate
predictions = clf.predict(test_embeddings)
accuracy = accuracy_score(test_labels, predictions)
print(f"Sentence-BERT + Logistic Regression Accuracy: {accuracy:.4f}")

### regression_model_e5-base.pkl

In [ ]:
import pickle

# Save model
with open('logistics.pkl', 'wb') as f:
    pickle.dump(clf, f)

## IntentEmbeddings

In [ ]:
import torch
torch.cuda.empty_cache()
from sentence_transformers import SentenceTransformer
from typing import Dict
import numpy as np
from functools import lru_cache

class SentenceEmbedder:
    def __init__(self):
        # Load a pre-trained Sentence-BERT model
        self.model = SentenceTransformer('intfloat/e5-base')  # Lightweight and effective

    def get_embedding(self, text):
       """
        Generate embedding for a given text, always prepending 'query: '.

        Args:
            text (str): The input string (treated as a query).

        Returns:
            np.ndarray: The embedding vector.
        """
        # Always prepend 'query: ' to the text
       formatted_text = "query: " + text.strip()
       return self.model.encode(formatted_text)


class IntentEmbeddings:
    def __init__(self, intents):
        self.intents = intents
        self.embedder = SentenceEmbedder()
        self.intent_embeddings = self.compute_intent_embeddings()

    def compute_intent_embeddings(self):
        """Generate embeddings for all intents."""
        embeddings = {}
        for intent, description in self.intents.items():
            combined_text = f"{intent}: {description}"  # Combine intent name and description
            embeddings[intent] = self.embedder.get_embedding(combined_text)
        return embeddings

## Customer agent

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import time

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
RATE_LIMIT_DELAY = 10
MODEL = "gpt-4o-mini"

def customer_agent(conversation_history, chatbot_question):
    system_prompt = f"""
You are playing the role of a real human user chatting with a support chatbot.

This is your conversation so far:
{conversation_history}

Now the chatbot is trying to understand your intent. It may ask you to choose between two or three possible options, or clarify your issue.
Respond like a real person would. Sometimes people are clear and direct and may choose option they find reasonable, sometimes they’re informal or unsure. Use your own words.
It's okay to be casual, make typos,or ramble a bit sometimes. Vary your behavior the way real users do.
Do not keep repreating same question again and again. Behave like user who wants to make chatbot understand their query.
If  chatbot responds with something completely irrelevant to your question inform it so.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chatbot_question}
    ]

    time.sleep(RATE_LIMIT_DELAY)
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.01
    )

    return response.choices[0].message.content.strip()


## DSMassFunction

In [ ]:
import torch
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher
from transformers import AutoModel, AutoTokenizer
import numpy as np
from datasets import load_dataset

dataset = load_dataset("clinc_oos", "plus")

# # Intent mapping
intent_feature = dataset["train"].features["intent"]
#index_to_name example: {0: 'restaurant_reviews', 1: 'nutrition_info', ..., 149: 'last_intent'}
index_to_name = {i: name for i, name in enumerate(intent_feature.names) if name != "oos"}


class DSMassFunction:
    def __init__(self, intent_embeddings, hierarchy, custom_thresholds=None):
        self.intent_embeddings = intent_embeddings
        self.embedder = SentenceEmbedder()
        self.hierarchy = hierarchy
        self.custom_thresholds = custom_thresholds or {}  # Use custom thresholds if provided
        self.conversation_history=[]
        self.user_response=None
        self.clf = clf
        self.index_to_name = index_to_name
    def is_leaf(self, intent):
        """Check if a node is a leaf in the hierarchy."""
        return len(self.hierarchy.get(intent, [])) == 0

    def get_threshold(self, intent):
        """Get threshold based on the level of the intent in the hierarchy."""
        if self.is_leaf(intent):
            return 0.1  # Minimum threshold for leaf nodes
        elif intent in self.hierarchy:  # Intermediate nodes
            return 0.2
        else:
            return 0.3  # Highest threshold for root nodes

    def get_confidence_threshold(self, intent):
        """Get confidence threshold for an intent. Use custom threshold if available, else default."""
        if intent in self.custom_thresholds:
            return self.custom_thresholds[intent]  # Use custom threshold
        elif self.is_leaf(intent):
            return 0.3  # Default for leaf nodes
        elif intent in self.hierarchy:  # Intermediate nodes
            return 0.5  # Default for intermediate nodes
        else:
            return 0.7  # Default for root nodes

    def get_node_depth(self, node):
        """
        Compute the depth of a node in the hierarchy.
        :param node: The node to compute the depth for.
        :return: The depth of the node (leaf nodes have the highest depth values).
        """
        depth = 0
        current = node
        # Traverse down the hierarchy until a leaf node is reached
        while current in self.hierarchy and self.hierarchy[current]:
            depth += 1
            current = self.hierarchy[current][0]  # Traverse to the first child
        return depth

    def find_lowest_common_ancestor(self, nodes):
        """
        Find the lowest common ancestor (LCA) of the given nodes in the hierarchy.
        :param nodes: List of nodes (intents) to find the LCA for.
        :return: The lowest common ancestor node.
        """
        if not nodes:
            return None

        # Find all ancestors for each node
        ancestors = []
        for node in nodes:
            node_ancestors = set()
            current = node
            while True:
                node_ancestors.add(current)
                parent = next((parent for parent, children in self.hierarchy.items() if current in children), None)
                if parent is None:
                    break
                current = parent
            ancestors.append(node_ancestors)

        # Find the intersection of all ancestors
        common_ancestors = set.intersection(*ancestors)

        # The LCA is the node with the minimum depth in the common ancestors
        lca = None
        min_depth = float('inf')
        for ancestor in common_ancestors:
            depth = self.get_node_depth(ancestor)
            if depth < min_depth:
                min_depth = depth
                lca = ancestor

        return lca

    def print_node_levels(self):
        """Print the level of each node in the hierarchy."""
        #print("Node Levels:")
        for intent in self.hierarchy:
            depth = self.get_node_depth(intent)
            #print(f"{intent}: Level {depth}")

    def compute_mass_function(self, user_query):
        """Compute normalized Dempster-Shafer mass function using clf probabilities."""
        self.conversation_history.append(f"User: {user_query}")

        # Get query embedding
        #embedder = SentenceEmbedder()
        query_embedding = self.embedder.get_embedding(user_query)

        # Get probabilities from clf
        probs = self.clf.predict_proba(query_embedding.reshape(1, -1))[0]

        # Initialize mass function
        mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}

        # Assign probabilities to dataset intents
        for idx, prob in zip(self.clf.classes_, probs):
            intent_name = self.index_to_name.get(int(idx), None)
            if intent_name and intent_name in mass_function:
                mass_function[intent_name] = prob

        # Normalize masses
        total_mass = sum(mass_function.values())
        if total_mass > 0:
            mass_function = {intent: mass / total_mass for intent, mass in mass_function.items()}
        else:
            # Handle zero mass case (unlikely with clf)
            mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
            mass_function["Uncertainty"] = 1.0

        #print(f"Mass Function: {mass_function}")
        return mass_function

    def compute_belief(self, mass_function):
        """Compute belief values for all intents in a bottom-up manner."""
        belief = {}

        def compute_node_belief(intent):
            # Use memoization to avoid recomputation
            if intent in belief:
                return belief[intent]

            # Start with the mass of the current node
            node_belief = mass_function.get(intent, 0)
            #print(f"Initial belief for {intent}: {node_belief}")

            # Add the belief of all child nodes
            children = self.hierarchy.get(intent, [])
            for child in children:
                child_belief = compute_node_belief(child)
                #print(f"Adding belief of {child} ({child_belief}) to {intent}")
                node_belief += child_belief

            # Cache the belief value for this node
            belief[intent] = node_belief
            #print(f"Final belief for {intent}: {node_belief}")
            return node_belief

        # Compute belief for all nodes in the hierarchy
        for intent in self.hierarchy:
            compute_node_belief(intent)

        return belief

    def combine_mass_functions(self, mass1, mass2):
        """Combine two mass functions using Dempster's rule."""
        combined_mass = {}
        conflict = 0

        for a in mass1:
            for b in mass2:
                # Find the highest common descendant (HCD) of a and b
                hcd = self.find_highest_common_descendant(a, b)

                # If HCD exists, use it as the intersection; otherwise, use "Uncertainty"
                intersection = hcd if hcd else "Uncertainty"

                contribution = mass1[a] * mass2[b]

                if intersection == "Uncertainty":
                    conflict += contribution
                else:
                    combined_mass[intersection] = combined_mass.get(intersection, 0) + contribution

        # Normalize to account for conflict
        if conflict < 1:
            for key in combined_mass:
                combined_mass[key] /= (1 - conflict)

        #print(f"Combined Mass Function: {combined_mass}")
        return combined_mass

    def find_highest_common_descendant(self, node1, node2):
        """
        Find the highest common descendant (HCD) of two nodes in the hierarchy.
        :param node1: The first node.
        :param node2: The second node.
        :return: The highest common descendant if it exists, otherwise None.
        """
        # Get all descendants of node1
        descendants1 = self.get_all_descendants(node1)

        # Get all descendants of node2
        descendants2 = self.get_all_descendants(node2)

        # Find the intersection of the two sets of descendants
        common_descendants = descendants1.intersection(descendants2)

        if not common_descendants:
            return None  # No common descendants

        # Find the highest common descendant (the one with the maximum depth)
        hcd = None
        max_depth = -1
        for descendant in common_descendants:
            depth = self.get_node_depth(descendant)
            if depth > max_depth:
                max_depth = depth
                hcd = descendant

        return hcd

    def get_all_descendants(self, node):
        """
        Get all descendants of a node in the hierarchy, including the node itself.
        :param node: The node to find descendants for.
        :return: A set of all descendants of the node, including the node itself.
        """
        descendants = set()
        stack = [node]

        while stack:
            current = stack.pop()
            descendants.add(current)  # Include the current node itself
            children = self.hierarchy.get(current, [])
            for child in children:
                stack.append(child)
        return descendants

    def evaluate_hierarchy(self, mass_function, current_level):
      """Evaluate the hierarchy and return nodes with belief above the threshold."""
      belief = self.compute_belief(mass_function)  # Precompute beliefs
      confident_nodes = []

      # Check all nodes at the current level
      for intent in current_level:
          intent_belief = belief.get(intent, 0)
          confidence_threshold = self.get_confidence_threshold(intent)  # Use custom thresholds

          if intent_belief >= confidence_threshold:
              confident_nodes.append((intent, intent_belief))

      # If confident nodes found, return them
      if confident_nodes:
          #print(f"Confident nodes at current level: {confident_nodes}")  # Debugging output

          # Filter leaf nodes
          confident_leaf_nodes = [node for node in confident_nodes if self.is_leaf(node[0])]

          if confident_leaf_nodes:
              #print(f"Confident leaf nodes: {confident_leaf_nodes}")  # Debugging output

              # Check if one leaf node has a significantly higher belief value
              if len(confident_leaf_nodes) > 1:
                  # Sort by belief value in descending order
                  confident_leaf_nodes.sort(key=lambda x: x[1], reverse=True)

                  # Check if the top node's belief is significantly higher than the second
                  top_node, top_belief = confident_leaf_nodes[0]
                  second_node, second_belief = confident_leaf_nodes[1]

                  # Define a threshold for "significantly higher" (e.g., 20% higher)
                  threshold = 0.4  # Adjust as needed
                  if (top_belief - second_belief) / top_belief > threshold:
                      print(f"Top confident leaf node has significantly higher belief: {top_node}")
                      return [(top_node, top_belief)], belief

              # If no single dominant leaf node, proceed with existing logic
              #print("No single dominant leaf node found. Proceeding with existing logic.")

          # If there is only one confident leaf node, return it
          if len(confident_leaf_nodes) == 1:
              print(f"Single confident leaf node found: {confident_leaf_nodes[0][0]}")
              return confident_leaf_nodes, belief

          # If multiple confident leaf nodes, group them by depth
          depth_groups = {}
          for intent, belief_value in confident_nodes:
              depth = self.get_node_depth(intent)
              if depth not in depth_groups:
                  depth_groups[depth] = []
              depth_groups[depth].append((intent, belief_value))

          # Find the subset of nodes at the lowest level (highest depth value)
          lowest_level = max(depth_groups.keys())  # Use max to find the lowest level
          lowest_level_nodes = depth_groups[lowest_level]

          # If there are multiple nodes at the lowest level, check if any are leaves
          if len(lowest_level_nodes) > 1:
              # Check if any of the nodes at the lowest level are leaves
              leaf_nodes = [node for node in lowest_level_nodes if self.is_leaf(node[0])]
              if leaf_nodes:
                  # If there are leaf nodes, return them
                  #print(f"Leaf nodes found at the lowest level: {leaf_nodes}")
                  return leaf_nodes, belief
              else:
                  # If no leaf nodes, find their LCA
                  lca = self.find_lowest_common_ancestor([intent for intent, _ in lowest_level_nodes])
                  if lca:
                      #print(f"Multiple confident nodes found at the lowest level. Lowest Common Ancestor: {lca}")
                      return [(lca, belief.get(lca, 0))], belief
          else:
              # If only one node at the lowest level, return it
              #print(f"Single confident node found at the lowest level: {lowest_level_nodes[0][0]}")
              return [lowest_level_nodes[0]], belief

          return confident_nodes, belief

      # Otherwise, move to the next level (parents of the current level nodes)
      next_level = []
      for node in current_level:
          # Find the parent of the current node
          parent = next((parent for parent, children in self.hierarchy.items() if node in children), None)
          if parent and parent not in next_level:  # Avoid duplicates
             next_level.append(parent)

      if not next_level:  # Base case: no more levels to explore
          return [], belief

      return self.evaluate_hierarchy(mass_function, next_level)

    def ask_clarification(self, parent_nodes, belief):
        """Generate clarification queries for ambiguous parent nodes."""
        clarification_queries = []

        for parent, _ in parent_nodes:
            children = self.hierarchy.get(parent, [])
            if len(children) < 4:
                # Directly ask about all children if they are fewer than 4
                clarification_queries.append((parent, children))
            else:
                # Sort children by belief and pick the top 3
                children_with_belief = [(child, belief.get(child, 0)) for child in children]
                children_with_belief.sort(key=lambda x: x[1], reverse=True)
                top_children = [child for child, _ in children_with_belief[:3]]
                clarification_queries.append((parent, top_children))

        return clarification_queries

    def evaluate_with_clarifications(self, initial_mass, depth=0, max_depth=5):
        """Evaluate recursively with clarification queries until a confident leaf is found."""
        if depth >= max_depth:
            return None, None

        def evaluate_from_leaves(current_mass, depth=0, max_depth=5):
            """Recursively evaluate the hierarchy starting from leaf nodes."""
            if depth >= max_depth:
                return None, None
            # Find all leaf nodes
            leaf_nodes = [intent for intent in self.hierarchy if self.is_leaf(intent)]
            #print(f"Leaf nodes: {leaf_nodes}")  # Debugging output

            # Evaluate the hierarchy starting from the leaf nodes
            confident_nodes, belief = self.evaluate_hierarchy(current_mass, leaf_nodes)

            if confident_nodes:
                # Check if there is exactly one confident leaf node
                confident_leaf_nodes = [node for node in confident_nodes if self.is_leaf(node[0])]
                if len(confident_leaf_nodes) == 1:
                    print(f"Single confident leaf node found: {confident_leaf_nodes[0][0]}")
                    return confident_leaf_nodes[0]

                # If multiple confident leaf nodes, find their LCA
                if len(confident_leaf_nodes) > 1:
                    lca = self.find_lowest_common_ancestor([intent for intent, _ in confident_leaf_nodes])
                    if lca:
                        #print(f"Multiple confident leaf nodes found. Lowest Common Ancestor: {lca}")

                        # Ask for clarification on the children of the LCA
                        #print(f"Asking for clarification on the children of the LCA: {lca}")
                        clarification_queries = self.ask_clarification([(lca, belief.get(lca, 0))], belief)

                        #print("Clarification Queries:")
                        for parent, children in clarification_queries:
                            #print(f"Parent: {parent}, Options: {children}")

                            while True:
                                chatbot_question = f"It seems like you're looking for something related to {parent}. Could you clarify which specific thing you're interested in from this category? Here are a few suggestions: ({children}). Does one of these sound like what you're looking for?"
                                self.conversation_history.append(f"Chatbot: {chatbot_question}")
                                self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                                #self.conversation_history.append(f"User: {self.user_response}")
                                #print(f"LLM:{self.user_response}")
                                user_mass = self.compute_mass_function(user_query=self.user_response)
                                break

                            # Combine with existing mass function
                            current_mass = self.combine_mass_functions(current_mass, user_mass)

                        # Continue evaluation with the updated mass function
                        return evaluate_from_leaves(current_mass, depth=depth + 1)
                    else:
                      options_no_parent = []
                      # Sort nodes by belief_value in descending order and pick top 3
                      top_nodes = sorted(confident_nodes, key=lambda x: x[1], reverse=True)[:3]
                      # Add only the intents to options_no_parent
                      for intent, belief_value in top_nodes:
                        options_no_parent.append(intent)
                      chatbot_question = f"There are quite a few different things that might match what you're looking for: ({options_no_parent}). Could you clarify a bit more to help me narrow it down?"
                      self.conversation_history.append(f"Chatbot: {chatbot_question}")
                      self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                      #self.conversation_history.append(f"User: {self.user_response}")
                      #print(f"LLM:{self.user_response}")
                      user_mass = self.compute_mass_function(user_query=self.user_response)
                      current_mass = self.combine_mass_functions(current_mass, user_mass)

                # Ask clarification queries for non-leaf nodes
                non_leaf_nodes = [(intent, belief_value) for intent, belief_value in confident_nodes if not self.is_leaf(intent)]
                clarification_queries = self.ask_clarification(non_leaf_nodes, belief)

                #print("Clarification Queries:")
                for parent, children in clarification_queries:
                    #print(f"Parent: {parent}, Options: {children}")

                    while True:
                        chatbot_question = f"t seems like you're looking for something related to {parent}. Could you clarify which specific thing you're interested in from this category? Here are a few suggestions: ({children}). Does one of these sound like what you're looking for?"
                        self.conversation_history.append(f"Chatbot: {chatbot_question}")
                        self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                        #self.conversation_history.append(f"User: {self.user_response}")
                        #print(f"LLM:{self.user_response}")
                        user_mass = self.compute_mass_function(user_query=self.user_response)
                        break

                    # Combine with existing mass function
                    current_mass = self.combine_mass_functions(current_mass, user_mass)

            else:
                chatbot_question = f"I’m not entirely sure what you're asking. Could you rephrase your question a bit? That would help me better understand what you're looking for!"
                self.conversation_history.append(f"Chatbot: {chatbot_question}")
                self.user_response =customer_agent("\n".join(self.conversation_history), chatbot_question)
                #self.conversation_history.append(f"User: {self.user_response}")
                #print(f"LLM:{self.user_response}")
                user_mass = self.compute_mass_function(user_query=self.user_response)
                current_mass = self.combine_mass_functions(current_mass, user_mass)

            # Recursively evaluate with the updated mass function
            return evaluate_from_leaves(current_mass,depth=depth + 1)

        # Start the evaluation from the leaf nodes with the initial mass
        return evaluate_from_leaves(initial_mass,depth=depth)

## hierarchical_intents

In [ ]:
hierarchical_intents = {
  "banking": "General queries related to banking services. Examples: 'How do I open a savings account?' 'What are the types of bank accounts available?'",
  "freeze_account": "Queries related to temporarily disabling or freezing a bank account. Examples: 'How can I freeze my account?' 'Can I freeze my bank account for security reasons?'",
  "routing": "Queries related to routing numbers or account numbers for transactions. Examples: 'What is my routing number?' 'How do I find my bank's routing number?'",
  "pin_change": "Requests to change the Personal Identification Number (PIN) for an account or card. Examples: 'How do I change my ATM pin?' 'Can I change my card PIN online?'",
  "bill_due": "Queries about upcoming bill payments or due dates. Examples: 'When is my electric bill due?' 'What's the due date for my water bill?'",
  "pay_bill": "Actions to pay a bill through a bank or service provider. Examples: 'Can you help me pay my credit card bill?' 'How do I pay my utility bill?'",
  "account_blocked": "Issues related to a blocked account. Examples: 'Why is my account blocked?' 'Can you unblock my account?'",
  "interest_rate": "Queries about current interest rates for savings or loan accounts. Examples: 'What's the interest rate for a savings account?' 'What is the interest rate on your mortgage loans?'",
  "min_payment": "Queries about the minimum payment due on a credit card or loan. Examples: 'What's the minimum payment on my credit card?' 'How much do I need to pay this month?'",
  "bill_balance": "Queries regarding the current balance on a bill or statement. Examples: 'What's my current bill balance?' 'How much do I owe on my account?'",
  "transfer": "Queries related to transferring money between accounts or to another person. Examples: 'Can I transfer money from my savings to checking account?' 'How do I send money to another person?'",
  "order_checks": "Requests to order new checks. Examples: 'How can I order new checks?' 'I need to request a new checkbook.'",
  "balance": "Queries about the current balance in an account. Examples: 'What's my bank account balance?' 'How much money do I have in my checking account?'",
  "spending_history": "Requests for past transactions or spending patterns. Examples: 'Can I see my recent spending history?' 'What were my last five transactions?'",
  "transactions": "Queries related to specific bank transactions. Examples: 'What was my last transaction?' 'Can you show me all transactions from this month?'",
  "report_fraud": "Reports or inquiries related to fraudulent activities on an account. Examples: 'I want to report fraud on my account.' 'My account has been hacked, what should I do?'",
  "credit_cards": "Queries or issues regarding credit cards. Examples: 'How do I apply for a credit card?' 'What's the limit on my credit card?'",
  "replacement_card_duration": "Questions about the duration to receive a replacement card. Examples: 'How long will it take to get a replacement card?' 'When will my new card arrive?'",
  "expiration_date": "Queries about the expiration date of a card or service. Examples: 'When does my credit card expire?' 'What's the expiration date on my debit card?'",
  "damaged_card": "Requests to report or replace a damaged card. Examples: 'I have a damaged card, how can I get a new one?' 'What should I do if my card is cracked?'",
  "improve_credit_score": "Tips or strategies for improving one's credit score. Examples: 'How can I improve my credit score?' 'What steps can I take to increase my credit rating?'",
  "report_lost_card": "Actions related to reporting a lost card. Examples: 'I lost my credit card, how do I report it?' 'What do I do if my debit card is lost?'",
  "card_declined": "Queries regarding declined card transactions. Examples: 'Why was my card declined?' 'What should I do if my card isn't working?'",
  "credit_limit_change": "Requests for increasing or decreasing a credit limit. Examples: 'Can I request a credit limit increase?' 'How do I lower my credit card limit?'",
  "apr": "Questions about the Annual Percentage Rate (APR) on loans or credit cards. Examples: 'What's the APR on my loan?' 'How can I get a lower APR on my credit card?'",
  "redeem_rewards": "Questions about redeeming rewards points or credits. Examples: 'How do I redeem my credit card rewards?' 'Can I use my rewards for a statement credit?'",
  "credit_limit": "Queries regarding the current credit limit available. Examples: 'What's my credit limit?' 'Can I check my available credit?'",
  "rewards_balance": "Checking the balance of rewards points or credits. Examples: 'How many rewards points do I have?' 'What's the balance on my rewards account?'",
  "application_status": "Inquiries about the status of an application (e.g., for a loan, credit card). Examples: 'What's the status of my credit card application?' 'Has my loan application been approved?'",
  "credit_score": "Questions regarding credit score information. Examples: 'What's my current credit score?' 'Can I check my credit score for free?'",
  "new_card": "Requests or inquiries about obtaining a new card. Examples: 'Can I apply for a new credit card?' 'How do I get a replacement debit card?'",
  "international_fees": "Queries related to fees for international transactions or services. Examples: 'What are the international fees for using my card?' 'How much do I get charged for overseas transactions?'",
  "kitchen_and_dining": "General queries regarding kitchen and dining products or services. Examples: 'What kitchen appliances do you recommend?' 'Can you suggest a good dining table?'",
  "food_last": "Questions about the shelf life or freshness of food. Examples: 'How long does this food last?' 'When does this frozen food expire?'",
  "confirm_reservation": "Confirming a reservation at a restaurant or service. Examples: 'Can you confirm my reservation for 7 PM?' 'Is my table reserved for tonight?'",
  "how_busy": "Asking about the current or future busyness of a place or event. Examples: 'How busy is the restaurant right now?' 'Is there a wait at the coffee shop?'",
  "ingredients_list": "Listing ingredients for a recipe or dish. Examples: 'What are the ingredients for a chocolate cake?' 'Can you give me the ingredients for a vegetarian lasagna?'",
  "calories": "Queries about the calorie content of food. Examples: 'How many calories are in a slice of pizza?' 'What's the calorie count for this smoothie?'",
  "nutrition_info": "Information about the nutritional content of food. Examples: 'What's the nutritional value of an apple?' 'Can you provide the nutrition facts for a salad?'",
  "recipe": "Requests for cooking recipes. Examples: 'Do you have a recipe for banana bread?' 'Can you give me a recipe for a vegan stir-fry?'",
  "restaurant_reviews": "Inquiries about reviews or ratings of a restaurant. Examples: 'What do people say about this restaurant?' 'Can you show me the reviews for this sushi place?'",
  "restaurant_reservation": "Making or managing reservations at restaurants. Examples: 'Can I reserve a table for two at 8 PM?' 'Can I change my reservation to 9 PM?'",
  "meal_suggestion": "Recommendations for meals based on preferences or dietary needs. Examples: 'What should I cook for dinner tonight?' 'Can you suggest a meal that's low in carbs?'",
  "restaurant_suggestion": "Recommendations for restaurants based on preferences. Examples: 'Can you suggest a good Italian restaurant near me?' 'Where should I go for sushi in the area?'",
  "cancel_reservation": "Cancelling an existing reservation at a restaurant or service. Examples: 'I need to cancel my reservation.' 'Can you cancel my booking for tomorrow night?'",
  "ingredient_substitution": "Finding substitutions for ingredients in a recipe. Examples: 'What can I use instead of eggs in this recipe?' 'Can I substitute almond milk for regular milk?'",
  "cook_time": "Information about how long to cook a certain dish or ingredient. Examples: 'How long does it take to cook rice?' 'What's the cook time for a roast chicken?'",
  "accept_reservations": "Asking if a place is currently accepting reservations. Examples: 'Are you accepting reservations for next week?' 'Can I book a table for two?'",
  "home": "Queries or requests related to home management, smart devices, or services. Examples: 'Can you turn off the lights?' 'Set the thermostat to 72°F.'",
  "what_song": "Asking about the current song being played or identifying a song. Examples: 'What song is playing right now?' 'Can you tell me the name of this song?'",
  "play_music": "Requests to play a specific song or playlist. Examples: 'Play my workout playlist.' 'Can you play some jazz music?'",
  "todo_list_update": "Adding, removing, or modifying items on a to-do list. Examples: 'Add buy groceries to my to-do list.' 'Remove the task 'call mom' from my list.'",
  "reminder": "Setting or asking for reminders for tasks or events. Examples: 'Remind me to call the doctor tomorrow.' 'Set a reminder for my meeting at 2 PM.'",
  "reminder_update": "Modifying or updating an existing reminder. Examples: 'Change the reminder for my dentist appointment to 3 PM.' 'Update my reminder to take medication at 9 AM.'",
  "calendar_update": "Making changes to calendar events or appointments. Examples: 'Can you update my meeting for next Monday?' 'Change my doctor's appointment to Thursday.'",
  "order_status": "Checking the status or delivery details of an order. Examples: 'What's the status of my recent order?' 'Has my package shipped yet?'",
  "update_playlist": "Modifying or adding to a music playlist. Examples: 'Add 'Bohemian Rhapsody' to my playlist.' 'Update my playlist with new songs.'",
  "shopping_list": "Creating, viewing, or managing a shopping list. Examples: 'Add eggs to my shopping list.' 'Show me my shopping list.'",
  "calendar": "Queries related to viewing or managing calendar events. Examples: 'What's on my calendar for today?' 'Show me my calendar for the week.'",
  "next_song": "Requesting to play the next song in a playlist or queue. Examples: 'Next song please!' 'Can you skip to the next track?'",
  "order": "Placing an order for goods or services. Examples: 'Order a pizza for delivery.' 'Can you place an order for a new laptop?'",
  "todo_list": "Viewing or managing a to-do list. Examples: 'What's on my to-do list today?' 'Can you check my to-do list?'",
  "shopping_list_update": "Updating or modifying an existing shopping list. Examples: 'Remove milk from my shopping list.' 'Add toothpaste to my shopping list.'",
  "smart_home": "Queries related to controlling smart home devices. Examples: 'Turn on the living room lights.' 'Set the temperature to 70°F.'",
  "auto_and_commute": "Questions or requests about car maintenance, commuting, or auto services. Examples: 'What time should I leave for my meeting to avoid traffic?' 'How do I check my tire pressure?'",
  "current_location": "Asking for the current geographical location. Examples: 'Where am I right now?' 'Can you tell me my current location?'",
  "oil_change_when": "Questions about when to change the oil in a vehicle. Examples: 'When should I get my oil changed?' 'How often should I change my car's oil?'",
  "oil_change_how": "Asking about how to change the oil in a car. Examples: 'How do I change the oil in my car?' 'Can I change my car's oil myself?'",
  "uber": "Inquiries related to Uber services, such as ride requests or fares. Examples: 'How much is an Uber ride to the airport?' 'Can you book an Uber for me?'",
  "traffic": "Queries about current or expected traffic conditions. Examples: 'How's the traffic on my route to work?' 'Is there heavy traffic on the highway?'",
  "tire_pressure": "Checking or inquiring about tire pressure. Examples: 'What should my tire pressure be?' 'How do I check the tire pressure on my car?'",
  "schedule_maintenance": "Requests for scheduling maintenance for vehicles or appliances. Examples: 'Can you schedule my car's maintenance?' 'I need to schedule maintenance for my washing machine.'",
  "gas": "Inquiries about fuel prices or fuel needs for a vehicle. Examples: 'What's the price of gas nearby?' 'How much gas do I have left in my car?'",
  "mpg": "Questions about a vehicle's miles per gallon (fuel efficiency). Examples: 'What's the MPG of my car?' 'How many miles per gallon does this car get?'",
  "distance": "Asking for distance between two locations or destinations. Examples: 'How far is it from here to the airport?' 'What's the distance from my home to the office?'",
  "directions": "Asking for driving or walking directions to a specific location. Examples: 'Can you give me directions to the nearest gas station?' 'How do I get to Main Street?'",
  "last_maintenance": "Checking the date or details of the last maintenance on a vehicle or appliance. Examples: 'When was the last time my car was serviced?' 'Can you show me the last maintenance record for my fridge?'",
  "gas_type": "Queries about the type of gas required for a vehicle. Examples: 'What type of gas does my car need?' 'Can I use regular gas in my car?'",
  "tire_change": "Questions or requests related to tire replacement. Examples: 'When should I change my tires?' 'How do I know if I need new tires?'",
  "jump_start": "Asking for advice or instructions on jump-starting a car. Examples: 'How do I jump-start my car?' 'Can you tell me how to use jumper cables?'",
  "travel": "General inquiries or requests related to travel plans or information. Examples: 'What's the best time to visit Paris?' 'Can you help me plan a trip to New York?'",
  "plug_type": "Asking about the type of plug or outlet needed for electrical devices in different countries. Examples: 'What plug type does Italy use?' 'Can I use my American charger in Europe?'",
  "travel_notification": "Notifications or alerts about upcoming travel. Examples: 'Please notify me about my flight status.' 'Can you send me a travel alert for my flight?'",
  "translate": "Asking for translations between different languages. Examples: 'Can you translate 'hello' into Spanish?' 'What's the German word for 'thank you'?'",
  "flight_status": "Checking the status of a flight, such as delays or cancellations. Examples: 'What's the status of Flight 567?' 'Is my flight on time?'",
  "international_visa": "Inquiries about visa requirements for international travel. Examples: 'Do I need a visa to visit France?' 'How do I apply for a tourist visa to the UK?'",
  "timezone": "Asking about time differences between locations or specific time zones. Examples: 'What's the time difference between New York and London?' 'What time zone is Tokyo in?'",
  "exchange_rate": "Questions about currency exchange rates. Examples: 'What's the exchange rate from USD to EUR?' 'How much is 100 yen worth in US dollars?'",
  "travel_suggestion": "Recommendations for travel destinations or tips. Examples: 'Where should I travel for a beach vacation?' 'Can you suggest a good city to visit in Europe?'",
  "travel_alert": "Notifications or alerts related to travel (e.g., weather, flight updates). Examples: 'Send me an alert about any changes to my flight.' 'Can you notify me of any travel advisories for Italy?'",
  "vaccines": "Questions about vaccination requirements for travel or health. Examples: 'Do I need any vaccinations to travel to Thailand?' 'What vaccines do I need before going to Africa?'",
  "lost_luggage": "Inquiries or reports regarding lost luggage during travel. Examples: 'My luggage is missing, what should I do?' 'Can you help me track my lost bag?'",
  "book_flight": "Requests to book a flight or search for flight availability. Examples: 'Can you book a flight to Los Angeles for next week?' 'Find me a flight to Paris.'",
  "book_hotel": "Inquiries or requests to book a hotel or check availability. Examples: 'Book a hotel for two nights in New York.' 'Is there a room available at this hotel?'",
  "carry_on": "Questions about carry-on luggage restrictions or policies. Examples: 'What's the size limit for a carry-on bag?' 'Can I bring a laptop in my carry-on?'",
  "car_rental": "Queries related to renting a car for travel. Examples: 'Can I rent a car in Los Angeles?' 'What's the price for a compact car rental?'",
  "utility": "Questions or requests related to utility services, such as electricity, water, or internet. Examples: 'How do I set up my water service?' 'Can I pay my electricity bill online?'",
  "weather": "Queries about the current or forecasted weather conditions. Examples: 'What's the weather like today?' 'Will it rain tomorrow?'",
  "alarm": "Setting or asking about alarms or reminders. Examples: 'Set an alarm for 7 AM.' 'What time is my alarm set for?'",
  "date": "Asking for the current date or day. Examples: 'What's today's date?' 'Can you tell me the date of my appointment?'",
  "find_phone": "Locating a lost phone or asking about phone location services. Examples: 'Can you help me find my phone?' 'Where is my phone right now?'",
  "share_location": "Sharing the current location with someone. Examples: 'Can you share my location with John?' 'Send my location to my friend.'",
  "timer": "Setting or managing a timer for a task. Examples: 'Set a timer for 30 minutes.' 'Can you start a timer for my workout?'",
  "make_call": "Making a phone call or requesting to call someone. Examples: 'Call my mom.' 'Can you dial the number for me?'",
  "calculator": "Performing mathematical calculations or conversions. Examples: 'What's 23 times 9?' 'Can you calculate the tip for a $50 meal?'",
  "definition": "Asking for the definition of a word or concept. Examples: 'What does 'ephemeral' mean?' 'Can you define 'quixotic'?'",
  "measurement_conversion": "Converting units of measurement (e.g., length, weight, temperature). Examples: 'How many kilometers is 5 miles?' 'Convert 100 grams to ounces.'",
  "flip_coin": "A request to simulate the flipping of a coin. Examples: 'Flip a coin.' 'Heads or tails?'",
  "spelling": "Asking how to spell a word. Examples: 'How do you spell 'accommodate'?' 'Can you spell 'restaurant'?'",
  "time": "Inquiries about the current time or time-related questions. Examples: 'What time is it?' 'What time does my meeting start?'",
  "roll_dice": "Rolling a die or simulating a dice roll. Examples: 'Roll a dice.' 'Can you roll two six-sided dice?'",
  "text": "Sending or composing a text message. Examples: 'Send a text to Mike.' 'Can you text my friend about the meeting?'",
  "work": "Queries related to work tasks, scheduling, or office-related matters. Examples: 'Can you schedule my meeting for 10 AM?' 'What's the status of my project?'",
  "pto_request_status": "Checking the status of Paid Time Off (PTO) requests. Examples: 'What's the status of my PTO request?' 'Has my vacation request been approved?'",
  "next_holiday": "Inquiring about the next upcoming holiday. Examples: 'When is the next public holiday?' 'What's the date of the next holiday?'",
  "insurance_change": "Questions about changes to insurance policies. Examples: 'Can I update my car insurance?' 'How do I change my health insurance provider?'",
  "insurance": "General inquiries about insurance policies or coverage. Examples: 'How much life insurance do I need?' 'What's covered under my car insurance?'",
  "meeting_schedule": "Inquiries about scheduling or managing meetings. Examples: 'Can you help me schedule a meeting for next Monday?' 'What time is my meeting tomorrow?'",
  "payday": "Questions related to payday or salary. Examples: 'When is my payday this month?' 'How much do I get paid next week?'",
  "taxes": "Questions about taxes or tax filing. Examples: 'What are the tax deadlines this year?' 'How do I file my taxes?'",
  "income": "Questions related to income, salary, or earnings. Examples: 'How much is my income tax refund?' 'Can you show me my income report?'",
  "rollover_401k": "Inquiries about rolling over a 401(k) account. Examples: 'How do I roll over my 401(k) to another account?' 'What are the benefits of rolling over a 401(k)?'",
  "pto_balance": "Checking the remaining balance of Paid Time Off (PTO). Examples: 'How many PTO hours do I have left?' 'What's my remaining vacation time?'",
  "pto_request": "Submitting or inquiring about a PTO request. Examples: 'Can I submit a PTO request for next week?' 'Has my PTO request been approved?'",
  "w2": "Questions about W2 forms or income reporting. Examples: 'When will my W2 be available?' 'How can I get my W2 form?'",
  "schedule_meeting": "Requesting to schedule a meeting or appointment. Examples: 'Can you schedule a meeting with my team for Friday?' 'What time works best for scheduling my meeting?'",
  "direct_deposit": "Queries related to setting up or managing direct deposit. Examples: 'How do I set up direct deposit?' 'When will my direct deposit go through?'",
  "pto_used": "Inquiries about how much PTO has been used. Examples: 'How much PTO have I used so far this year?' 'Can I see my PTO usage history?'",
  "small_talk": "Casual conversation or light-hearted exchanges. Examples: 'What's your favorite color?' 'Do you have any hobbies?'",
  "who_made_you": "Asking about the creator or origin of the AI. Examples: 'Who created you?' 'Who made you?'",
  "meaning_of_life": "Philosophical questions about life or existence. Examples: 'What is the meaning of life?' 'Why are we here?'",
  "who_do_you_work_for": "Questions about the AI's employer or purpose. Examples: 'Who do you work for?' 'Are you working for anyone specific?'",
  "do_you_have_pets": "Casual or personal questions about the AI's 'personal life'. Examples: 'Do you have any pets?' 'What kind of pets do you like?'",
  "what_are_your_hobbies": "Asking about the AI's interests or hobbies. Examples: 'What do you like to do for fun?' 'What are your hobbies?'",
  "fun_fact": "Requesting a fun or interesting fact. Examples: 'Tell me a fun fact!' 'Can you share an interesting fact?'",
  "what_is_your_name": "Asking for the name of the AI. Examples: 'What's your name?' 'Can you tell me your name?'",
  "where_are_you_from": "Asking where the AI 'comes from'. Examples: 'Where are you from?' 'Where did you originate?'",
  "goodbye": "Saying goodbye or ending the conversation. Examples: 'Goodbye!' 'See you later!'",
  "thank_you": "Expressing gratitude. Examples: 'Thank you for your help!' 'Thanks a lot!'",
  "greeting": "A friendly greeting or acknowledgment. Examples: 'Hello!' 'Hi there!'",
  "tell_joke": "Asking the AI to tell a joke. Examples: 'Tell me a joke!' 'Can you make me laugh?'",
  "are_you_a_bot": "Inquiring if the AI is a bot. Examples: 'Are you a robot?' 'Is this a bot talking to me?'",
  "how_old_are_you": "Asking for the AI's age. Examples: 'How old are you?' 'When were you created?'",
  "what_can_i_ask_you": "Asking what kinds of questions can be asked. Examples: 'What kinds of things can I ask you?' 'What do you know?'",
  "meta": "Queries related to system or functionality settings of the AI. Examples: 'Can you change your voice?' 'Can you speak another language?'",
  "change_speed": "Changing the speed at which the AI speaks or responds. Examples: 'Can you speak slower?' 'Change your speaking speed.'",
  "user_name": "Inquiries or requests about the user's name. Examples: 'What's my name?' 'Can you remember my name?'",
  "whisper_mode": "Changing the AI's mode to whisper or speak softly. Examples: 'Can you speak in whisper mode?' 'Change to whisper mode.'",
  "yes": "Confirmation or affirmation of something. Examples: 'Is this the correct answer?' 'Yes, that's right.'",
  "change_volume": "Adjusting the volume of the AI's output. Examples: 'Can you lower the volume?' 'Increase the volume, please.'",
  "no": "Denial or negation of something. Examples: 'Do you agree with this?' 'No, that's not correct.'",
  "change_language": "Changing the language the AI uses. Examples: 'Can you speak French?' 'Change to Spanish.'",
  "repeat": "Requesting the AI to repeat something. Examples: 'Can you repeat that?' 'Say that again.'",
  "change_accent": "Changing the accent or style of speech. Examples: 'Can you speak with a British accent?' 'Change your accent to Australian.'",
  "cancel": "Cancelling an action or command. Examples: 'Cancel my reminder.' 'Stop the timer.'",
  "sync_device": "Syncing or connecting a device to the system. Examples: 'Sync my phone with the app.' 'Can you connect to my Bluetooth speaker?'",
  "change_user_name": "Changing the user's name on the system. Examples: 'Can you change my name to Alex?' 'Set my name to John.'",
  "change_ai_name": "Changing the name of the AI. Examples: 'Can you call yourself Alexa?' 'Rename the assistant to Max.'",
  "reset_settings": "Resetting the system or preferences to default. Examples: 'Reset my settings to default.' 'Can you restore the original settings?'",
  "maybe": "Expressing uncertainty or possibility. Examples: 'Do you think this is a good idea?' 'Maybe, let me think about it.'"
}

## hierarchy

In [ ]:
hierarchy ={
    "banking": [
        "freeze_account",
        "routing",
        "pin_change",
        "bill_due",
        "pay_bill",
        "account_blocked",
        "interest_rate",
        "min_payment",
        "bill_balance",
        "transfer",
        "order_checks",
        "balance",
        "spending_history",
        "transactions",
        "report_fraud"
    ],
    "freeze_account": [],
    "routing": [],
    "pin_change": [],
    "bill_due": [],
    "pay_bill": [],
    "account_blocked": [],
    "interest_rate": [],
    "min_payment": [],
    "bill_balance": [],
    "transfer": [],
    "order_checks": [],
    "balance": [],
    "spending_history": [],
    "transactions": [],
    "report_fraud": [],
    "credit_cards": [
        "replacement_card_duration",
        "expiration_date",
        "damaged_card",
        "improve_credit_score",
        "report_lost_card",
        "card_declined",
        "credit_limit_change",
        "apr",
        "redeem_rewards",
        "credit_limit",
        "rewards_balance",
        "application_status",
        "credit_score",
        "new_card",
        "international_fees"
    ],
    "replacement_card_duration": [],
    "expiration_date": [],
    "damaged_card": [],
    "improve_credit_score": [],
    "report_lost_card": [],
    "card_declined": [],
    "credit_limit_change": [],
    "apr": [],
    "redeem_rewards": [],
    "credit_limit": [],
    "rewards_balance": [],
    "application_status": [],
    "credit_score": [],
    "new_card": [],
    "international_fees": [],
    "kitchen_and_dining": [
        "food_last",
        "confirm_reservation",
        "how_busy",
        "ingredients_list",
        "calories",
        "nutrition_info",
        "recipe",
        "restaurant_reviews",
        "restaurant_reservation",
        "meal_suggestion",
        "restaurant_suggestion",
        "cancel_reservation",
        "ingredient_substitution",
        "cook_time",
        "accept_reservations"
    ],
    "food_last": [],
    "confirm_reservation": [],
    "how_busy": [],
    "ingredients_list": [],
    "calories": [],
    "nutrition_info": [],
    "recipe": [],
    "restaurant_reviews": [],
    "restaurant_reservation": [],
    "meal_suggestion": [],
    "restaurant_suggestion": [],
    "cancel_reservation": [],
    "ingredient_substitution": [],
    "cook_time": [],
    "accept_reservations": [],
    "home": [
        "what_song",
        "play_music",
        "todo_list_update",
        "reminder",
        "reminder_update",
        "calendar_update",
        "order_status",
        "update_playlist",
        "shopping_list",
        "calendar",
        "next_song",
        "order",
        "todo_list",
        "shopping_list_update",
        "smart_home"
    ],
    "what_song": [],
    "play_music": [],
    "todo_list_update": [],
    "reminder": [],
    "reminder_update": [],
    "calendar_update": [],
    "order_status": [],
    "update_playlist": [],
    "shopping_list": [],
    "calendar": [],
    "next_song": [],
    "order": [],
    "todo_list": [],
    "shopping_list_update": [],
    "smart_home": [],
    "auto_and_commute": [
        "current_location",
        "oil_change_when",
        "oil_change_how",
        "uber",
        "traffic",
        "tire_pressure",
        "schedule_maintenance",
        "gas",
        "mpg",
        "distance",
        "directions",
        "last_maintenance",
        "gas_type",
        "tire_change",
        "jump_start"
    ],
    "current_location": [],
    "oil_change_when": [],
    "oil_change_how": [],
    "uber": [],
    "traffic": [],
    "tire_pressure": [],
    "schedule_maintenance": [],
    "gas": [],
    "mpg": [],
    "distance": [],
    "directions": [],
    "last_maintenance": [],
    "gas_type": [],
    "tire_change": [],
    "jump_start": [],
    "travel": [
        "plug_type",
        "travel_notification",
        "translate",
        "flight_status",
        "international_visa",
        "timezone",
        "exchange_rate",
        "travel_suggestion",
        "travel_alert",
        "vaccines",
        "lost_luggage",
        "book_flight",
        "book_hotel",
        "carry_on",
        "car_rental"
    ],
    "plug_type": [],
    "travel_notification": [],
    "translate": [],
    "flight_status": [],
    "international_visa": [],
    "timezone": [],
    "exchange_rate": [],
    "travel_suggestion": [],
    "travel_alert": [],
    "vaccines": [],
    "lost_luggage": [],
    "book_flight": [],
    "book_hotel": [],
    "carry_on": [],
    "car_rental": [],
    "utility": [
        "weather",
        "alarm",
        "date",
        "find_phone",
        "share_location",
        "timer",
        "make_call",
        "calculator",
        "definition",
        "measurement_conversion",
        "flip_coin",
        "spelling",
        "time",
        "roll_dice",
        "text"
    ],
    "weather": [],
    "alarm": [],
    "date": [],
    "find_phone": [],
    "share_location": [],
    "timer": [],
    "make_call": [],
    "calculator": [],
    "definition": [],
    "measurement_conversion": [],
    "flip_coin": [],
    "spelling": [],
    "time": [],
    "roll_dice": [],
    "text": [],
    "work": [
        "pto_request_status",
        "next_holiday",
        "insurance_change",
        "insurance",
        "meeting_schedule",
        "payday",
        "taxes",
        "income",
        "rollover_401k",
        "pto_balance",
        "pto_request",
        "w2",
        "schedule_meeting",
        "direct_deposit",
        "pto_used"
    ],
    "pto_request_status": [],
    "next_holiday": [],
    "insurance_change": [],
    "insurance": [],
    "meeting_schedule": [],
    "payday": [],
    "taxes": [],
    "income": [],
    "rollover_401k": [],
    "pto_balance": [],
    "pto_request": [],
    "w2": [],
    "schedule_meeting": [],
    "direct_deposit": [],
    "pto_used": [],
    "small_talk": [
        "who_made_you",
        "meaning_of_life",
        "who_do_you_work_for",
        "do_you_have_pets",
        "what_are_your_hobbies",
        "fun_fact",
        "what_is_your_name",
        "where_are_you_from",
        "goodbye",
        "thank_you",
        "greeting",
        "tell_joke",
        "are_you_a_bot",
        "how_old_are_you",
        "what_can_i_ask_you"
    ],
    "who_made_you": [],
    "meaning_of_life": [],
    "who_do_you_work_for": [],
    "do_you_have_pets": [],
    "what_are_your_hobbies": [],
    "fun_fact": [],
    "what_is_your_name": [],
    "where_are_you_from": [],
    "goodbye": [],
    "thank_you": [],
    "greeting": [],
    "tell_joke": [],
    "are_you_a_bot": [],
    "how_old_are_you": [],
    "what_can_i_ask_you": [],
    "meta": [
        "change_speed",
        "user_name",
        "whisper_mode",
        "yes",
        "change_volume",
        "no",
        "change_language",
        "repeat",
        "change_accent",
        "cancel",
        "sync_device",
        "change_user_name",
        "change_ai_name",
        "reset_settings",
        "maybe"
    ],
    "change_speed": [],
    "user_name": [],
    "whisper_mode": [],
    "yes": [],
    "change_volume": [],
    "no": [],
    "change_language": [],
    "repeat": [],
    "change_accent": [],
    "cancel": [],
    "sync_device": [],
    "change_user_name": [],
    "change_ai_name": [],
    "reset_settings": [],
    "maybe": []
}


## Optimal Thresholds

## load dataset

In [ ]:
from datasets import load_dataset
from pathlib import Path
import json
from pprint import pprint

# --- Load the dataset ---
dataset = load_dataset("clinc_oos", "plus")
train_data = dataset["train"]

# --- Load intent map ---
intent_feature = dataset["train"].features["intent"]
intent_map = {i: name for i, name in enumerate(intent_feature.names) if name != "oos"}

# --- Filter and reformat the dataset ---
formatted_data = []
# Iterate directly over the train_data Dataset object
for ex in train_data:
    # Print the current example object to see its structure
    # Check if the intent is not 'oos' (intent == 42)
    if ex["intent"] != 42:
        # Append the reformatted data if the intent is in-scope
        formatted_data.append({"Example": ex["text"], "Label": intent_map[ex["intent"]]})


# --- Print a few samples ---
print(formatted_data[:5])

In [ ]:
custom_thresholds={}

### load regression_model_e5-base_0.88.pkl as clf

In [ ]:
import pickle
with open('logistics.pkl', 'rb') as f:
    clf = pickle.load(f)

### Split the data into 4 parts

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Function to split a list into n equal parts
def chunk_data(data, n_chunks):
    avg_chunk_size = len(data) // n_chunks
    chunks = []
    for i in range(n_chunks):
        start_idx = i * avg_chunk_size
        end_idx = (i + 1) * avg_chunk_size if i != n_chunks - 1 else len(data)
        chunks.append(data[start_idx:end_idx])
    return chunks

# Create a dictionary to store belief values for all examples
belief_results = []

# Split the data into 4 parts
n_chunks = 4
chunks = chunk_data(formatted_data, n_chunks)

# Assuming `intent_embedder` and `ds_calculator` are already initialized as:
intent_embedder = IntentEmbeddings(hierarchical_intents)
ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy)

# Process each chunk of data
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i + 1}...")

    chunk_belief_results = []

    for row in tqdm(chunk, desc=f"Chunk {i + 1}", unit="row"):
        user_query = row["Example"]
        correct_intent = row["Label"]

        try:
            mass_function = ds_calculator.compute_mass_function(user_query)
            belief = ds_calculator.compute_belief(mass_function)

            if belief:
                chunk_belief_results.append({
                    "user_question": user_query,
                    "correct_intent": correct_intent,
                    "belief_values": belief
                })
            else:
                print(f"Warning: No belief values for query: {user_query}")

        except Exception as e:
            print(f"Error processing query '{user_query}': {e}")
    # Convert the chunk results to a DataFrame
    belief_df_chunk = pd.DataFrame(chunk_belief_results)

    # Save the results to a CSV file (name based on chunk number)
    output_filename = f"e5-base_train_clinc_trial_chunk_{i + 1}.csv"
    belief_df_chunk.to_csv(output_filename, index=False)

    print(f"Chunk {i + 1} saved to {output_filename}")

print("All chunks processed and saved successfully!")

### Merged all chunks into train_clinc_trial

In [ ]:
import pandas as pd

# File names for the chunks
chunk_files = [
    "train_clinc_trial_chunk_1.csv",
    "train_clinc_trial_chunk_2.csv",
    "train_clinc_trial_chunk_3.csv",
    "train_clinc_trial_chunk_4.csv"
]

# Load and concatenate all chunks
dfs = [pd.read_csv(file) for file in chunk_files]
merged_df = pd.concat(dfs, ignore_index=True)

# Save merged file (optional but useful)
merged_df.to_csv("e5-base_train_clinc_trial.csv", index=False)

print("Merged all chunks into train_clinc_trial.csv")


###  Parse the belief values

In [ ]:
import pandas as pd
import re

# Load merged CSV
df = pd.read_csv("train_clinc_trial.csv")

def parse_belief_values(raw_str):
    pattern = r"'(.*?)': (?:np\.float64\()?([0-9.eE+-]+)\)?"
    matches = re.findall(pattern, raw_str)
    return {intent: float(val) for intent, val in matches}

# Parse the belief values safely
df["belief_values"] = df["belief_values"].apply(parse_belief_values)

# Convert belief_values dicts to a new DataFrame all at once
belief_df = df["belief_values"].apply(pd.Series).fillna(0.0)

# Merge into original and drop the raw column
df = pd.concat([df.drop(columns=["belief_values"]), belief_df], axis=1)

# Save the transformed DataFrame
df.to_csv("transformed_train_clinc_trial.csv", index=False)

# Optional preview
print(df.head())


In [ ]:
import pandas as pd

# Load the original CSV
df = pd.read_csv("transformed_train_clinc_trial.csv")

# Define or load hierarchy
# Example:
# with open("hierarchy.json") as f:
#     hierarchy = json.load(f)

# Get all unique intents
intents = set(hierarchy.keys())
for children in hierarchy.values():
    intents.update(children)
intents = sorted(intents)

# Function to get ancestors
def get_ancestors(intent, hierarchy):
    ancestors = set()
    stack = [intent]
    while stack:
        current = stack.pop()
        ancestors.add(current)
        for parent, children in hierarchy.items():
            if current in children:
                stack.append(parent)
    return ancestors

# Create a list of dictionaries for new columns
new_columns = []
for index, row in df.iterrows():
    correct_intent = row["correct_intent"]
    ancestors = get_ancestors(correct_intent, hierarchy)
    new_row = {f"is_correct_{intent}": int(intent in ancestors) for intent in intents}
    new_columns.append(new_row)

from collections import defaultdict

def get_descendants(intent, hierarchy):
    children = hierarchy.get(intent, [])
    all_descendants = set(children)
    for child in children:
        all_descendants.update(get_descendants(child, hierarchy))
    return all_descendants

# Step 2: For each ancestor, sum the values of its descendant columns
for ancestor in hierarchy.keys():
    descendants = get_descendants(ancestor, hierarchy)
    # Only include descendants that actually exist as columns in the DataFrame
    valid_descendants = [desc for desc in descendants if desc in df.columns]
    if valid_descendants:
        df[f"sum_descendants_{ancestor}"] = df[valid_descendants].sum(axis=1)
# Convert new columns to DataFrame and concatenate
new_df = pd.DataFrame(new_columns)
df = pd.concat([df.drop(columns=["correct_intent"]), new_df], axis=1)

# Save to new CSV
df.to_csv("updated_train_clinc_trial.csv", index=False)
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

# Load the dataset
df = pd.read_csv("updated_train_clinc_trial.csv")

# Generate list of all intents from the columns
intents = sorted([col.replace("is_correct_", "") for col in df.columns if col.startswith("is_correct_")])

# Dictionary to store optimal thresholds and scores
optimal_results = {}

# Precompute thresholds from 0.00 to 1.00
thresholds = np.linspace(0, 1, 101)

# Iterate over each intent
for intent in intents:
    belief_values = df[intent].values
    ground_truth = df[f"is_correct_{intent}"].values

    best_threshold = 0.0
    best_weighted_f1 = 0.0

    for threshold in thresholds:
        predictions = (belief_values >= threshold).astype(int)
        weighted_f1 = f1_score(ground_truth, predictions, average="weighted")

        if weighted_f1 > best_weighted_f1:
            best_weighted_f1 = weighted_f1
            best_threshold = threshold

    optimal_results[intent] = {
        "threshold": best_threshold,
        "f1_score": best_weighted_f1
    }

# Print the optimal thresholds and scores
print("Optimal Thresholds and Weighted F1 Scores:")
for intent, result in optimal_results.items():
    print(f"{intent}: Threshold = {result['threshold']:.2f}, F1 Score = {result['f1_score']:.4f}")

In [ ]:
# Save the optimal thresholds and scores to a JSON file
with open("optimal_thresholds_clinc.json", "w") as json_file:
    json.dump(optimal_results, json_file, indent=4)

# Print the saved thresholds and scores
print("Optimal Thresholds and Weighted F1 Scores saved to 'optimal_thresholds_clinc.json':")
for intent, result in optimal_results.items():
    print(f"{intent}: Threshold = {result['threshold']:.2f}")

In [ ]:
import json

# Load the original JSON structure
with open('optimal_thresholds_clinc.json', 'r') as f:
    full_data = json.load(f)

# Extract the simplified format
custom_thresholds = {intent: data["threshold"] for intent, data in full_data.items()}


# Print a few to preview
print(dict(list(custom_thresholds .items())))

## Main

In [ ]:
def main():
    from datasets import load_dataset
    import pandas as pd

    # Initialize intent embeddings
    intent_embedder = IntentEmbeddings(hierarchical_intents)

    # Load dataset and define index_to_name
    dataset = load_dataset("clinc_oos", "plus")
    intent_feature = dataset["train"].features["intent"]
    index_to_name = {i: name for i, name in enumerate(intent_feature.names) if name != "oos"}

    # Load and filter test data
    test_data = dataset["test"]
    val_in_scope = [ex for ex in test_data if ex["intent"] != 42]  # exclude oos

    # Split into 10 chunks
    total = len(val_in_scope)
    chunk_size = total // 10
    chunks = [val_in_scope[i*chunk_size : (i+1)*chunk_size] for i in range(9)]
    chunks.append(val_in_scope[9*chunk_size:])  # final chunk includes remainder
    ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy, custom_thresholds)
    # Process each chunk
    for idx, val_chunk in enumerate(chunks):
        print(f"\n--- Processing chunk {idx + 1}/10 ({len(val_chunk)} samples) ---")

        results = []

        for ex in val_chunk:
            user_query = ex["text"]
            true_intent_idx = ex["intent"]
            true_intent = index_to_name.get(true_intent_idx, "unknown")
            ds_calculator.conversation_history = []
            initial_mass = ds_calculator.compute_mass_function(user_query)
            DS_prediction = ds_calculator.evaluate_with_clarifications(initial_mass)
            results.append({
                "query": user_query,
                "prediction": str(DS_prediction),
                "true_intent": true_intent,
                "interaction": ds_calculator.conversation_history
            })

            print(ds_calculator.conversation_history)

        # Save chunk results
        df = pd.DataFrame(results)
        df.to_csv(f"predictions_test_chunk_{idx}_e5-base_match_temp_prompt_improve=0.05.csv", index=False)
        print(f"Saved chunk {idx} to predictions_test_chunk_{idx}.csv")
if __name__ == "__main__":
    main()


In [ ]:
import socket
socket.gethostbyname("api.openai.com")


## Compute accuracy

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score
import re
import numpy as np
import os

def compute_accuracy_from_chunks():
    # Generate file paths for chunks 0 to 9
    base_filename= "predictions_test_chunk_{}_clinc_logistics.csv"
    chunk_files = [base_filename.format(i) for i in range(10)]
    dfs = [pd.read_csv(file) for file in chunk_files]
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv("predictions_test_clinc_logistics", index=False)
    # Check if files exist
    missing_files = [f for f in chunk_files if not os.path.exists(f)]
    if missing_files:
        print(f"Error: Missing chunk files: {missing_files}")
    if len(chunk_files) != 10:
        print(f"Warning: Expected 10 chunk files, found {len(chunk_files)}")

    # Parse prediction
    def parse_prediction(pred_str):
        if not isinstance(pred_str, str):
            return "unknown", None
        try:
            # Extract intent and probability from ('intent', np.float64(prob))
            match = re.match(r"\('([^']+)',\s*np\.float64\(([\d.]+)\)\)", pred_str)
            if match:
                intent = match.group(1).lower().strip()
                prob = float(match.group(2))
                return intent, prob
            return "unknown", None
        except Exception as e:
            print(f"Warning: Failed to parse prediction: {pred_str}, error: {e}")
            return "unknown", None

    # Aggregate results
    all_predictions = []
    all_true_labels = []
    all_correct_probs = []
    all_incorrect_probs = []
    total_correct = 0
    total = 0
    per_chunk_metrics = []

    # Process each chunk
    for chunk_file in chunk_files:
        print(f"\nProcessing chunk: {chunk_file}")
        try:
            df = pd.read_csv(chunk_file)
        except FileNotFoundError:
            print(f"Error: {chunk_file} not found")
            continue
        except pd.errors.EmptyDataError:
            print(f"Error: {chunk_file} is empty")
            continue

        # Verify columns
        expected_columns = ["query", "prediction", "true_intent"]
        if not all(col in df.columns for col in expected_columns):
            print(f"Error: {chunk_file} missing required columns. Found: {list(df.columns)}")
            continue

        # Track chunk-specific metrics
        chunk_predictions = []
        chunk_true_labels = []
        chunk_correct_probs = []
        chunk_incorrect_probs = []
        chunk_correct = 0
        chunk_total = 0
        predicted_counts = {}
        true_counts = {}

        # Debug: print first few rows
        print(f"First 5 rows of {chunk_file}:")
        print(df.head().to_string())
        print()

        # Process rows
        for index, row in df.iterrows():
            predicted_intent, prob = parse_prediction(row["prediction"])
            true_intent = str(row["true_intent"]).lower().strip()

            chunk_total += 1
            total += 1
            chunk_predictions.append(predicted_intent)
            chunk_true_labels.append(true_intent)
            all_predictions.append(predicted_intent)
            all_true_labels.append(true_intent)

            # Update counts
            predicted_counts[predicted_intent] = predicted_counts.get(predicted_intent, 0) + 1
            true_counts[true_intent] = true_counts.get(true_intent, 0) + 1

            # Compare
            is_correct = predicted_intent == true_intent
            if is_correct:
                chunk_correct += 1
                total_correct += 1
                if prob is not None:
                    chunk_correct_probs.append(prob)
                    all_correct_probs.append(prob)
            else:
                if prob is not None:
                    chunk_incorrect_probs.append(prob)
                    all_incorrect_probs.append(prob)

            # Debug: print first 10 and matches
            if index < 10 or is_correct:
                print(f"Row {index}: Predicted: {predicted_intent}, True: {true_intent}, Prob: {prob}, Match: {is_correct}")

        # Debug: print counts
        print(f"\nPredicted intent counts in {chunk_file}:")
        for intent, count in sorted(predicted_counts.items()):
            print(f"{intent}: {count}")
        print(f"\nTrue intent counts in {chunk_file}:")
        for intent, count in sorted(true_counts.items()):
            print(f"{intent}: {count}")

        # Compute chunk metrics
        chunk_accuracy = chunk_correct / chunk_total if chunk_total > 0 else 0.0
        chunk_f1 = f1_score(chunk_true_labels, chunk_predictions, average='macro') if chunk_predictions else 0.0
        chunk_avg_correct_prob = np.mean(chunk_correct_probs) if chunk_correct_probs else 0.0
        chunk_avg_incorrect_prob = np.mean(chunk_incorrect_probs) if chunk_incorrect_probs else 0.0

        print(f"\nChunk Metrics for {chunk_file}:")
        print(f"Accuracy: {chunk_accuracy:.4f} ({chunk_correct}/{chunk_total} correct)")
        print(f"Macro-averaged F1 Score: {chunk_f1:.4f}")
        print(f"Average probability for correct predictions: {chunk_avg_correct_prob:.4f} ({len(chunk_correct_probs)} examples)")
        print(f"Average probability for incorrect predictions: {chunk_avg_incorrect_prob:.4f} ({len(chunk_incorrect_probs)} examples)")

        per_chunk_metrics.append({
            'file': chunk_file,
            'accuracy': chunk_accuracy,
            'f1': chunk_f1,
            'correct_prob': chunk_avg_correct_prob,
            'incorrect_prob': chunk_avg_incorrect_prob,
            'correct_count': chunk_correct,
            'total': chunk_total
        })

    # Compute overall metrics
    overall_accuracy = total_correct / total if total > 0 else 0.0
    overall_f1 = f1_score(all_true_labels, all_predictions, average='macro') if all_predictions else 0.0
    overall_avg_correct_prob = np.mean(all_correct_probs) if all_correct_probs else 0.0
    overall_avg_incorrect_prob = np.mean(all_incorrect_probs) if all_incorrect_probs else 0.0

    print("\nOverall Metrics Across All Chunks:")
    print(f"Total utterances processed: {total}")
    print(f"Accuracy: {overall_accuracy:.4f} ({total_correct}/{total} correct)")
    print(f"Macro-averaged F1 Score: {overall_f1:.4f}")
    print(f"Average probability for correct predictions: {overall_avg_correct_prob:.4f} ({len(all_correct_probs)} examples)")
    print(f"Average probability for incorrect predictions: {overall_avg_incorrect_prob:.4f} ({len(all_incorrect_probs)} examples)")

    # Summary of per-chunk metrics
    print("\nPer-Chunk Summary:")
    for metric in per_chunk_metrics:
        print(f"{metric['file']}: Accuracy={metric['accuracy']:.4f}, F1={metric['f1']:.4f}, "
              f"Correct Prob={metric['correct_prob']:.4f}, Incorrect Prob={metric['incorrect_prob']:.4f}")

compute_accuracy_from_chunks()

## Analysis of Results

In [ ]:
import pandas as pd

merged_file = "predictions_test_clinc_logistics.csv"
# Read the CSV (only the 'interaction' column)
df = pd.read_csv(merged_file, usecols=['interaction'])

# Count occurrences of "chatbot:" and track rows
total_count = sum(str(interaction).count("Chatbot:") for interaction in df['interaction'] if pd.notna(interaction))
num_rows = sum(1 for interaction in df['interaction'] if pd.notna(interaction))

# Calculate average
average_per_row = total_count / num_rows if num_rows > 0 else 0

# Print results
print(f"Total occurrences of 'chatbot:': {total_count}")
print(f"Rows processed: {num_rows}")
print(f"Average 'chatbot:' per row: {average_per_row:.4f}")

In [ ]:
# Filter in-scope queries
dataset = load_dataset("clinc_oos", "plus")
train_data = dataset["train"]
test_data = dataset["test"]

# Filter in-scope queries
train_in_scope = [ex for ex in train_data if ex["intent"] != 42]
test_in_scope = [ex for ex in test_data if ex["intent"] != 42]

# Load Sentence-BERT
model = SentenceTransformer('intfloat/e5-base')

# Generate embeddings
train_texts = [ex["text"] for ex in train_in_scope]
test_texts = [ex["text"] for ex in test_in_scope]
test_embeddings = model.encode(test_texts, batch_size=64, show_progress_bar=True)
train_labels = [ex["intent"] for ex in train_in_scope]
test_labels = [ex["intent"] for ex in test_in_scope]
# Predict and evaluate
predictions = clf.predict(test_embeddings)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from statsmodels.stats.contingency_tables import mcnemar

# Load the dataset from the CSV file (the one you mentioned earlier)
data = pd.read_csv('predictions_test_clinc_logistics.csv')

# Extract true labels and model predictions
true_labels = data['true_intent'].values
print(true_labels[0])
# Manually encode the true labels to numeric values using the label_to_int dictionary

# Extract your model's predictions from the CSV (first element of the tuple in 'prediction' column)
model_predictions = data['prediction'].apply(lambda x: eval(x)[0]).values
print(model_predictions[0])
# Manually encode the model predictions to numeric values using the label_to_int dictionary
# Make predictions using the logistic model
logistic_predictions = clf.predict(test_embeddings)
logistic_predictions = [index_to_name[idx] for idx in logistic_predictions]
print(logistic_predictions[0])

# Create the contingency table for the McNemar test
# b = Number of instances where SVM is correct and your model is wrong
# c = Number of instances where your model is correct and SVM is wrong
b = np.sum((logistic_predictions == true_labels) & (model_predictions!= true_labels))
c = np.sum((logistic_predictions != true_labels) & (model_predictions == true_labels))

# Construct the contingency table
contingency_table = np.array([[0, b], [c, 0]])

# Perform McNemar test
result = mcnemar(contingency_table, exact=True)

# Print the result
print(f"McNemar test result: statistic={result.statistic}, p-value={result.pvalue}")
